In [1]:
import pandas as pd
import numpy as np
import glob
import os
import shutil
import matplotlib.pyplot as plt

import mne
import pyprep 

from mne.preprocessing import read_ica
from mne_icalabel import label_components
from mne_bids.write import _write_json

from mne_bids import BIDSPath, read_raw_bids, get_entity_vals, write_raw_bids

%matplotlib widget

In [2]:
# Define the path to the BIDS root directory
bids_root = '/scratch2/alinat/project/PD-EEG-ANON_eegOnly/sourcedataRest'

# Define the path to the output directory (derivatives)
out_path = '/scratch2/alinat/project/PD-EEG-ANON_eegOnly/derivatives/mne_pipeline_eye-closed_run1'

# Get all subjects and sessions from the BIDS dataset
subjects = get_entity_vals(bids_root, 'subject')
sessions = get_entity_vals(bids_root, 'session')

# Define the task
task = 'restEyesClosed'

In [ ]:
# Loop through each subject and session
for subject in subjects:
    subject_id = f"sub-{subject}"

    for session in sessions:
        session_id = f"ses-{session}"
        session_path = os.path.join(out_path, subject_id, session_id, 'eeg')

        # Define file paths for the current session
        ica_path = os.path.join(session_path, f"{subject_id}_{session_id}_task-{task}_proc-ica_ica.fif")
        tsv_path = os.path.join(session_path, f"{subject_id}_{session_id}_task-{task}_proc-ica_components.tsv")
        json_path = os.path.join(session_path, f"{subject_id}_{session_id}_task-{task}_proc-ica_components.json")
        epo_path = os.path.join(session_path, f"{subject_id}_{session_id}_task-{task}_epo.fif")

        # Check if the ICA file exists
        if not os.path.isfile(ica_path):
            #print(f"ICA file not found: {ica_path}")
            continue

        try:
            # Read ICA object and epochs
            ica = read_ica(ica_path)
            epochs = mne.read_epochs(epo_path, preload=True)

            # Ensure epochs are rereferenced to average
            #if not epochs.info['custom_ref_applied']:
            #    epochs.set_eeg_reference('average', projection=False)

            # Apply ICLabel to label components
            out = label_components(epochs, ica, method='iclabel')
            print(f"Labels for {subject_id}, {session_id}: {out['labels']}")

            # Identify bad components
            bad_components = [i for i, label in enumerate(out['labels']) if label not in ['brain', 'other']]
            ica.exclude.extend(bad_components)

            # Save updated ICA object
            ica.save(ica_path, overwrite=True)

            # Update the TSV file
            tsv_data = pd.read_csv(tsv_path, sep='\t')
            tsv_data['status'] = ['bad' if i in bad_components else 'good' for i in range(len(out['labels']))]
            tsv_data['status_description'] = [f'mne_icalabel::{label}' if i in bad_components else 'n/a' for i, label in enumerate(out['labels'])]
            tsv_data['ic_type'] = out['labels']
            tsv_data['annotate_author'] = 'Alina'
            tsv_data['annotate_method'] = 'mne_icalabel'
            tsv_data.to_csv(tsv_path, sep="\t", index=False, encoding="utf-8")

            # Update the JSON file
            component_json = {
                "annotate_method": "Method used for annotating components (e.g., manual, iclabel)",
                "annotate_author": "The name of the person who ran the annotation",
                "ic_type": "The type of annotation (e.g., brain, muscle artifact, eye blink, etc.)",
            }
            _write_json(json_path, component_json, overwrite=True)

            print(f"Processed {subject_id}, {session_id}")

        except Exception as e:
            print(f"Failed to process {subject_id}, {session_id}: {e}")

print("Finished processing all subjects and sessions.")